In [1]:
import pyspark 
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.master("local")\
        .appName("djsbalakrishnan-demo")\
        .getOrCreate()

# for all hight level API's we do not use sparkcontext 
# we rather use our spark session 
# inside spark session, we have spark context, sql context and other objects 

25/07/30 17:24:21 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/30 17:24:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
spark

In [5]:
# read input from movies 
# read input from ratings 

movieDF = spark.read.format("csv").option("delimiter", ",").option("header", True)\
        .option("inferschema", True).load("/user/userjuly2025019/spark/movies.csv")

ratingsDF = spark.read.format("csv").option("delimiter", ",").option("header", True)\
        .option("inferschema", True).load("/user/userjuly2025019/spark/ratings.csv")


In [6]:
movieDF.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)



In [7]:
ratingsDF.printSchema()

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: double (nullable = true)
 |-- timestamp: integer (nullable = true)



In [8]:
movieDF.createOrReplaceTempView("movies") 
ratingsDF.createOrReplaceTempView("rating")

# it is creating a view behind the scenes 
# view is a temporary table 

In [14]:
spark.sql("select * from movies limit 10").show(truncate=False)

+-------+----------------------------------+-------------------------------------------+
|movieId|title                             |genres                                     |
+-------+----------------------------------+-------------------------------------------+
|1      |Toy Story (1995)                  |Adventure|Animation|Children|Comedy|Fantasy|
|2      |Jumanji (1995)                    |Adventure|Children|Fantasy                 |
|3      |Grumpier Old Men (1995)           |Comedy|Romance                             |
|4      |Waiting to Exhale (1995)          |Comedy|Drama|Romance                       |
|5      |Father of the Bride Part II (1995)|Comedy                                     |
|6      |Heat (1995)                       |Action|Crime|Thriller                      |
|7      |Sabrina (1995)                    |Comedy|Romance                             |
|8      |Tom and Huck (1995)               |Adventure|Children                         |
|9      |Sudden Death

In [12]:
spark.sql("select * from rating limit 10").show(truncate=False)

+------+-------+------+----------+
|userId|movieId|rating|timestamp |
+------+-------+------+----------+
|1     |2      |3.5   |1112486027|
|1     |29     |3.5   |1112484676|
|1     |32     |3.5   |1112484819|
|1     |47     |3.5   |1112484727|
|1     |50     |3.5   |1112484580|
|1     |112    |3.5   |1094785740|
|1     |151    |4.0   |1094785734|
|1     |223    |4.0   |1112485573|
|1     |253    |4.0   |1112484940|
|1     |260    |4.0   |1112484826|
+------+-------+------+----------+



In [17]:
# Write the SQL to get the movies with most number of ratings in comedy genre 
highest_no_rating_df = spark.sql(
    """
    select m.title, count(*) as rating_count 
    from movies m 
    join rating r 
    on m.movieId = r.movieId 
    where m.genres like '%Comedy%' 
    group by m.title 
    order by rating_count desc 
    limit 1
    """
)

In [16]:
highest_no_rating_df.show()

+-------------------+------------+
|              title|rating_count|
+-------------------+------------+
|Pulp Fiction (1994)|       67310|
+-------------------+------------+



In [18]:
highest_no_rating_df.explain(True)

== Parsed Logical Plan ==
'GlobalLimit 1
+- 'LocalLimit 1
   +- 'Sort ['rating_count DESC NULLS LAST], true
      +- 'Aggregate ['m.title], ['m.title, 'count(1) AS rating_count#167]
         +- 'Filter 'm.genres LIKE %Comedy%
            +- 'Join Inner, ('m.movieId = 'r.movieId)
               :- 'SubqueryAlias m
               :  +- 'UnresolvedRelation [movies], [], false
               +- 'SubqueryAlias r
                  +- 'UnresolvedRelation [rating], [], false

== Analyzed Logical Plan ==
title: string, rating_count: bigint
GlobalLimit 1
+- LocalLimit 1
   +- Sort [rating_count#167L DESC NULLS LAST], true
      +- Aggregate [title#17], [title#17, count(1) AS rating_count#167L]
         +- Filter genres#18 LIKE %Comedy%
            +- Join Inner, (movieId#16 = movieId#39)
               :- SubqueryAlias m
               :  +- SubqueryAlias movies
               :     +- Relation[movieId#16,title#17,genres#18] csv
               +- SubqueryAlias r
                  +- SubqueryAlia